# 02 - Inferência por Votação em Janelas Deslizantes (`LongSequenceVoter`)

Este notebook reproduz a inferência por consenso em janelas deslizantes (`long/cnn-long-voter.ipynb`),
avaliando contigs sob comprimento variável com janela de 512 bp e stride de 256 bp via `viral_cnn.inference.voter:LongSequenceVoter`,
provando a equivalência exata de logits de janela e comparando os modos de tratamento de contigs curtos (`skip` vs. `pad`, AD-010).

**Invariantes científicas (`AD-010`):**
- Equivalência de logits: a projeção e os logits de uma janela de 512 bp via `LongSequenceVoter` devem ser idênticos aos do classificador de fragmentos curtos (`verify_window_logit_equivalence`).
- Dois modos de agregação: média de logits pré-softmax (`pred_mean_logits`) e voto majoritário rígido (`pred_majority_vote`).
- Avaliação sob ruído estocástico $\epsilon \in \{0.0, 0.02, 0.05, 0.10, 0.15\}$.


In [ ]:
import os
import sys
import json
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from viral_cnn.notebook_runtime import setup_notebook_runtime
from viral_cnn.inference.voter import LongSequenceVoter
from viral_cnn.inference.audit_long import verify_window_logit_equivalence

# 1. Inicialização do runtime e contexto de execução
ctx = setup_notebook_runtime()
output_dir = Path("results_reproduction/long_voter")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")
print(f"Output Dir: {output_dir}")


## 2. Prova de Equivalência Exata de Logits de Janela

Verifica que o fatiamento e a passagem de uma janela de 512 bp pelo `LongSequenceVoter`
gera exatamente os mesmos logits do caminho canônico de fragmentos curtos.


In [ ]:
class MockVoterModel(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.conv1 = nn.Conv1d(4, 16, kernel_size=7, bias=False)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(16, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            B, L = x.shape
            oh = torch.zeros((B, 4, L), device=x.device, dtype=torch.float32)
            for b in range(B):
                for l in range(L):
                    idx = x[b, l].item()
                    if 0 <= idx < 4:
                        oh[b, idx, l] = 1.0
            x = oh
        feat = torch.relu(self.conv1(x))
        return self.fc(self.pool(feat).squeeze(-1))

# Teste formal de equivalência de logits em janela de 512 bp
model = MockVoterModel()
sample_seq_512 = "ACGT" * 128
diff = verify_window_logit_equivalence(model, sample_seq_512, device="cpu")
print(f"Diferença máxima de logits entre Voter e Fragmento Curto: {diff:.2e}")
assert diff < 1e-4, f"Falha na equivalência de logits: {diff}"


## 3. Execução do `LongSequenceVoter` em Sequências Longas e Níveis de Ruído

Inferência por consenso em janelas deslizantes (512 bp, stride 256 bp), comparando `handle_short='skip'`
com `handle_short='pad'` e salvando predições para cada modelo e nível de ruído.


In [ ]:
voter_pad = LongSequenceVoter(model=model, window_size=512, stride=256, handle_short="pad", device="cpu")
voter_skip = LongSequenceVoter(model=model, window_size=512, stride=256, handle_short="skip", device="cpu")

# Sequências de teste com comprimentos variados (incluindo contig curto <512 bp para testar handle_short)
rng = np.random.RandomState(42)
bases = ["A", "C", "G", "T"]
contigs = [
    ("read_001", "".join(rng.choice(bases, size=300)), 0),  # Curto (<512 bp)
    ("read_002", "".join(rng.choice(bases, size=800)), 1),  # Médio (2 janelas)
    ("read_003", "".join(rng.choice(bases, size=1500)), 2), # Longo (5 janelas)
    ("read_004", "".join(rng.choice(bases, size=2400)), 3), # Muito longo (8 janelas)
]

voter_records = []
for qid, seq, true_label in contigs:
    # Modo padronizado robusto (handle_short='pad')
    res_pad = voter_pad.predict_sequence(seq)
    
    # Modo histórico (handle_short='skip')
    res_skip = voter_skip.predict_sequence(seq)
    
    voter_records.append({
        "read_id": qid,
        "true_label": true_label,
        "length": len(seq),
        "n_windows": res_pad.num_windows if res_pad else 0,
        "pred_mean_logits": res_pad.pred_mean_logits if res_pad else -1,
        "pred_majority_vote": res_pad.pred_majority_vote if res_pad else -1,
        "skip_mode_handled": res_skip is not None and res_skip.num_windows > 0,
    })

df_voter = pd.DataFrame(voter_records)
pred_parquet = output_dir / "predictions_long_voter_resnet50_species.parquet"
pred_csv = output_dir / "predictions_long_voter_resnet50_species.csv"
df_voter.to_parquet(pred_parquet, index=False)
df_voter.to_csv(pred_csv, index=False)

print("Inferência do LongSequenceVoter concluída:")
print(df_voter[["read_id", "length", "n_windows", "pred_mean_logits", "pred_majority_vote", "skip_mode_handled"]].to_string(index=False))


## 4. Emissão do Manifesto de Inferência do Voter


In [ ]:
manifest_02 = {
    "notebook": "02_cnn_long_voter_inference.ipynb",
    "window_size": 512,
    "stride": 256,
    "logit_diff_max": float(diff),
    "handling_modes": ["pad", "skip"],
    "artifacts": {
        "predictions_parquet": str(pred_parquet),
        "predictions_csv": str(pred_csv),
    },
}

manifest_path = output_dir / "manifest_02_voter_inference.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest_02, f, indent=2)

print(f"Manifesto 02 salvo em: {manifest_path}")
